In [1]:
import json
import pandas as pd
import os
from groq import Groq
import time
import time
import google.generativeai as genai
import os
import matplotlib.pyplot as plt
from openai import OpenAI
import google.generativeai as genai
from biochatter.llm_connect import GptConversation
import warnings
from biochirp_agent import biochirp_agent_output
warnings.filterwarnings('ignore')


/tmp/ipykernel_505229/2363448217.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
/home/abhishekh/anaconda3/envs/biochirp/lib/python3.11/site-packages/biochatter/llm_connect/conversation.py:34: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydan

In [2]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

In [3]:
number_of_question = 100
number_of_trial = 3
VALID_OPTIONS = {"A", "B", "C", "D", "E"}

In [4]:
with open("../../resources/prompts/mcq_evaluator.md", "r", encoding="utf-8") as f:
    prompt_md = f.read()


In [5]:
rows = []
with open("../data/MedQA/test.jsonl") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(rows)

# Expand options dict into columns
options_df = df["options"].apply(pd.Series)

# Rename option columns
options_df = options_df.rename(columns=lambda c: f"option_{c}")

# Merge back
df_structured = pd.concat(
    [df.drop(columns=["options"]), options_df],
    axis=1
)


df_structured = df_structured.head(number_of_question)
df_structured.reset_index(drop=True, inplace=True)
# df_structured["LLM_output"] = None

df_structured.head(1)

,question,answer,meta_info,answer_idx,option_A,option_B,option_C,option_D,option_E
0,A junior orthopaedic surgery resident is compl...,Tell the attending that he cannot fail to disc...,step1,C,Disclose the error to the patient but leave it...,Disclose the error to the patient and put it i...,Tell the attending that he cannot fail to disc...,Report the physician to the ethics committee,Refuse to dictate the operative report


In [30]:
# import importlib
# import biochirp_agent
# importlib.reload(biochirp_agent)

##### Result of Biochirp agent

In [31]:
model = "biochirp"
df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    

    for i in range(number_of_question):

        start_time = time.perf_counter()

        question = df_structured_tmp.loc[i, "question"]

        print(f"Running for question {i}: {question}")

        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]

        



        answer = await biochirp_agent_output(
                    user_prompt:=f"""Question: {question}
                            Options:
                            A. {option_A}
                            B. {option_B}
                            C. {option_C}
                            D. {option_D}
                            E. {option_E}
                            """,
                    system_prompt=prompt_md,
                    judge_model = "gpt-5-mini",
                    # max_output_tokens = 1,
                )
        end_time = time.perf_counter()

        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )

    df_structured_tmp[f"is_correct_run_{run_number}"] = (
        df_structured_tmp["answer_idx"]
        == df_structured_tmp[f"LLM_output_run_{run_number}"]
    )

    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_biochirp = df_structured_tmp.copy(deep=True)
df_structured_biochirp["model"] = model



Running for question 0: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
{'model': 'llama-3.3-70b-versatile', 'answer': 'B', 'error': None}
{'model': 'gpt-5-nano', 'answer': 'B', 'error': None}
{'model': 'gemini-2.5-flash-lite', 'answer': 'B', 'error': None}
{'model': 'grok-4-1-fast-non-reasoning-latest', 'answer': 'B', 'error': None}
Running for question 1: A 67-year-old man with transitional cell carcinoma of the bladder comes to the 

##### Result of gpt-5-mini

In [ ]:
model = "gpt-5-mini"

client = OpenAI()

# client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    for i in range(number_of_question):

        start_time = time.perf_counter()

        question = df_structured_tmp.loc[i, "question"]
        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]

        print(f"Running for question {i}: {question}")

        chat_completion = client.chat.completions.create(
            model=model,
            # temperature=0,
            # max_tokens=1,
            messages=[
                {"role": "system", "content": prompt_md},
                {"role": "user", "content":f"""Question: {question}
                                Options:
                                A. {option_A}
                                B. {option_B}
                                C. {option_C}
                                D. {option_D}
                                E. {option_E}
                                """
                                }
            ]
        )

        

        answer = chat_completion.choices[0].message.content.strip().upper()

        end_time = time.perf_counter()

        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )

    df_structured_tmp[f"is_correct_run_{run_number}"] = (
        df_structured_tmp["answer_idx"]
        == df_structured_tmp[f"LLM_output_run_{run_number}"]
    )

    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_gpt_5_mini = df_structured_tmp.copy(deep=True)
df_structured_gpt_5_mini["model"] = model



Running for question 0: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Running for question 1: A 67-year-old man with transitional cell carcinoma of the bladder comes to the physician because of a 2-day history of ringing sensation in his ear. He received this first course of neoadjuvant chemotherapy 1 week ago. Pure tone audiometry shows a sensorineural hearing loss of 45 dB. The expected beneficial effect of the drug that caused thi

##### Result of gpt-4.1-nano

In [ ]:
model = "gpt-4.1-nano"

client = OpenAI()

# client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    for i in range(number_of_question):

        start_time = time.perf_counter()

        question = df_structured_tmp.loc[i, "question"]
        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]

        print(f"Running for question {i}: {question}")

        chat_completion = client.chat.completions.create(
            model=model,
            # temperature=0,
            # max_tokens=1,
            messages=[
                {"role": "system", "content": prompt_md},
                {"role": "user", "content":f"""Question: {question}
                                Options:
                                A. {option_A}
                                B. {option_B}
                                C. {option_C}
                                D. {option_D}
                                E. {option_E}
                                """
                                }
            ]
        )

        

        answer = chat_completion.choices[0].message.content.strip().upper()

        end_time = time.perf_counter()

        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )

    df_structured_tmp[f"is_correct_run_{run_number}"] = (
        df_structured_tmp["answer_idx"]
        == df_structured_tmp[f"LLM_output_run_{run_number}"]
    )

    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_gpt_41_nano = df_structured_tmp.copy(deep=True)
df_structured_gpt_41_nano["model"] = model



Running for question 0: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Running for question 1: A 67-year-old man with transitional cell carcinoma of the bladder comes to the physician because of a 2-day history of ringing sensation in his ear. He received this first course of neoadjuvant chemotherapy 1 week ago. Pure tone audiometry shows a sensorineural hearing loss of 45 dB. The expected beneficial effect of the drug that caused thi

##### Result from gpt-5-nano

In [ ]:
model = "gpt-5-nano"

client = OpenAI()

# client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    for i in range(number_of_question):

        start_time = time.perf_counter()

        question = df_structured_tmp.loc[i, "question"]
        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]

        print(f"Running for question {i}: {question}")

        chat_completion = client.chat.completions.create(
            model=model,
            # temperature=0,
            # max_tokens=1,
            messages=[
                {"role": "system", "content": prompt_md},
                {"role": "user", "content":f"""Question: {question}
                                Options:
                                A. {option_A}
                                B. {option_B}
                                C. {option_C}
                                D. {option_D}
                                E. {option_E}
                                """
                                }
            ]
        )

        

        answer = chat_completion.choices[0].message.content.strip().upper()

        end_time = time.perf_counter()

        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )

    df_structured_tmp[f"is_correct_run_{run_number}"] = (
        df_structured_tmp["answer_idx"]
        == df_structured_tmp[f"LLM_output_run_{run_number}"]
    )

    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_gpt_5_nano = df_structured_tmp.copy(deep=True)
df_structured_gpt_5_nano["model"] = model



Running for question 0: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Running for question 1: A 67-year-old man with transitional cell carcinoma of the bladder comes to the physician because of a 2-day history of ringing sensation in his ear. He received this first course of neoadjuvant chemotherapy 1 week ago. Pure tone audiometry shows a sensorineural hearing loss of 45 dB. The expected beneficial effect of the drug that caused thi

##### Result of llama-3.3-70b-versatile

In [ ]:
model = "llama-3.3-70b-versatile"
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    for i in range(number_of_question):

        start_time = time.perf_counter()

        question = df_structured_tmp.loc[i, "question"]
        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]

        print(f"Running for question {i}: {question}")

        chat_completion = client.chat.completions.create(
            model=model,
            temperature=0,
            top_p=1,
            max_tokens=1,
            messages=[
                {"role": "system", "content": prompt_md},
                {
                    "role": "user",
                    "content": f"""Question: {question}
Options:
A. {option_A}
B. {option_B}
C. {option_C}
D. {option_D}
E. {option_E}
"""
                }
            ],
        )

        

        answer = chat_completion.choices[0].message.content.strip().upper()

        end_time = time.perf_counter()

        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )

    df_structured_tmp[f"is_correct_run_{run_number}"] = (
        df_structured_tmp["answer_idx"]
        == df_structured_tmp[f"LLM_output_run_{run_number}"]
    )

    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_llama_70b_versatile = df_structured_tmp.copy(deep=True)
df_structured_llama_70b_versatile["model"] = model



Running for question 0: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Running for question 1: A 67-year-old man with transitional cell carcinoma of the bladder comes to the physician because of a 2-day history of ringing sensation in his ear. He received this first course of neoadjuvant chemotherapy 1 week ago. Pure tone audiometry shows a sensorineural hearing loss of 45 dB. The expected beneficial effect of the drug that caused thi

##### Result for gemini-2.5-flash-lite

In [ ]:
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

model_name = "gemini-2.5-flash-lite"

client = genai.GenerativeModel(
    model_name,
    system_instruction=prompt_md
)

df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    for i in range(number_of_question):

        start_time = time.perf_counter()
        question = df_structured_tmp.loc[i, "question"]
        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]



        print(f"Running for question {i}: {question}")

        response = client.generate_content(
            f"""Question: {question}
                Options:
                A. {option_A}
                B. {option_B}
                C. {option_C}
                D. {option_D}
                E. {option_E}
"""
        )

        

        answer = response.text.strip().upper()

        end_time = time.perf_counter()

        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )

    df_structured_tmp[f"is_correct_run_{run_number}"] = (
        df_structured_tmp["answer_idx"]
        == df_structured_tmp[f"LLM_output_run_{run_number}"]
    )

    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_gemini = df_structured_tmp.copy(deep=True)
df_structured_gemini["model"] = model_name


Running for question 0: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Running for question 1: A 67-year-old man with transitional cell carcinoma of the bladder comes to the physician because of a 2-day history of ringing sensation in his ear. He received this first course of neoadjuvant chemotherapy 1 week ago. Pure tone audiometry shows a sensorineural hearing loss of 45 dB. The expected beneficial effect of the drug that caused thi

##### Result of biochatter(gpt-4.1-nano)

In [13]:
model = "gpt-4.1-nano"

df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    for i in range(number_of_question):

        conversation = GptConversation(
            model_name=model,
            prompts={}
        )
        conversation.append_system_message(prompt_md)

        success = conversation.set_api_key(os.getenv("OPENAI_API_KEY"), user="my_user")

        question = df_structured_tmp.loc[i, "question"]
        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]


        print(f"Running for question {i}: {question}")

        start_time = time.perf_counter()


        answer, token_usage, correction = conversation.query(
            f"""Question: {question}
                Options:
                A. {option_A}
                B. {option_B}
                C. {option_C}
                D. {option_D}
                E. {option_E}
                """
        )

        end_time = time.perf_counter()

        print(f"Answer is {answer}")

        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )

    df_structured_tmp[f"is_correct_run_{run_number}"] = (
        df_structured_tmp["answer_idx"]
        == df_structured_tmp[f"LLM_output_run_{run_number}"]
    )

    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_biochatter_41_nano = df_structured_tmp.copy(deep=True)
df_structured_biochatter_41_nano["model"] = "biochatter(gpt-4.1-nano)"




2026-02-03 22:46:30.617 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:30.702 
  command:

    streamlit run /home/abhishekh/anaconda3/envs/biochirp/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-02-03 22:46:30.702 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:30.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:30.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:30.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Running for question 0: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Answer is C


2026-02-03 22:46:32.140 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:32.141 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:32.141 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:32.142 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:32.144 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Running for question 1: A 67-year-old man with transitional cell carcinoma of the bladder comes to the physician because of a 2-day history of ringing sensation in his ear. He received this first course of neoadjuvant chemotherapy 1 week ago. Pure tone audiometry shows a sensorineural hearing loss of 45 dB. The expected beneficial effect of the drug that caused this patient's symptoms is most likely due to which of the following actions?
Answer is D


2026-02-03 22:46:33.808 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:33.809 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:33.810 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:33.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:46:33.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


...

Running for question 97: A 26-year-old male presents to his primary care physician with complaints of burning with urination, penile discharge, and intermittent fevers. A urethral smear shows gram negative diplococci within white blood cells. The organism grows well when cultured on Thayer-Martin agar. The patient is prescribed a course of ceftriaxone and the infection resolves without further complication. One year later, the patient returns with the same infection. Which of the following best explains this lack of lasting immunity?
Answer is B


2026-02-03 22:53:37.700 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:53:37.701 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:53:37.701 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:53:37.702 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:53:37.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Running for question 98: A 37-year-old man with no significant past medical history is rear-ended in a motor vehicle accident. He reported significant neck pain to emergency responders, but otherwise denies weakness, numbness or tingling in his extremities. His vitals on presentation to the ED are HR 90, BP 140/80, RR 20, SpO2 98%. What is the most appropriate next step upon presentation to the emergency room?
Answer is C


2026-02-03 22:53:38.915 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:53:38.916 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:53:38.916 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:53:38.917 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:53:38.918 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Running for question 99: A 43-year-old man with a history of schizophrenia, currently controlled with medication, comes in for an appointment with his internist. He is concerned about abnormal discharge from both nipples over the past 3 months. The fluid is white, and he wonders if it could be milk. On further review of systems, he endorses a diminished sexual drive. The physician suspects that one of the patient's medications may be the culprit for these symptoms. Which of the following medications is NOT likely to be the cause?
Answer is C
Accuracy of Run 3 is 0.6800


##### Result of biochatter(gpt-5-nano)

In [14]:
model = "gpt-5-nano"

df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    for i in range(number_of_question):

        conversation = GptConversation(
            model_name=model,
            prompts={}
        )

        conversation.append_system_message(prompt_md)

        success = conversation.set_api_key(os.getenv("OPENAI_API_KEY"), user="my_user")


        question = df_structured_tmp.loc[i, "question"]
        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]



        print(f"Running for question {i}: {question}")


        start_time = time.perf_counter()

        answer, token_usage, correction = conversation.query(
            f"""Question: {question}
                Options:
                A. {option_A}
                B. {option_B}
                C. {option_C}
                D. {option_D}
                E. {option_E}
                """
        )

        end_time = time.perf_counter()

        print(f"Answer is {answer}")

        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )

    df_structured_tmp[f"is_correct_run_{run_number}"] = (
        df_structured_tmp["answer_idx"]
        == df_structured_tmp[f"LLM_output_run_{run_number}"]
    )

    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_biochatter_5_nano = df_structured_tmp.copy(deep=True)
df_structured_biochatter_5_nano["model"] = "biochatter(gpt-5-nano)"

2026-02-03 22:56:59.724 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:56:59.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:56:59.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:56:59.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:56:59.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Running for question 0: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Answer is B


2026-02-03 22:57:16.160 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:57:16.161 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:57:16.162 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:57:16.163 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:57:16.163 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Running for question 1: A 67-year-old man with transitional cell carcinoma of the bladder comes to the physician because of a 2-day history of ringing sensation in his ear. He received this first course of neoadjuvant chemotherapy 1 week ago. Pure tone audiometry shows a sensorineural hearing loss of 45 dB. The expected beneficial effect of the drug that caused this patient's symptoms is most likely due to which of the following actions?
Answer is E


2026-02-03 22:57:20.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:57:20.840 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:57:20.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:57:20.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 22:57:20.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


...

Running for question 97: A 26-year-old male presents to his primary care physician with complaints of burning with urination, penile discharge, and intermittent fevers. A urethral smear shows gram negative diplococci within white blood cells. The organism grows well when cultured on Thayer-Martin agar. The patient is prescribed a course of ceftriaxone and the infection resolves without further complication. One year later, the patient returns with the same infection. Which of the following best explains this lack of lasting immunity?
Answer is B


2026-02-03 23:56:29.467 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 23:56:29.468 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 23:56:29.469 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 23:56:29.470 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 23:56:29.471 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Running for question 98: A 37-year-old man with no significant past medical history is rear-ended in a motor vehicle accident. He reported significant neck pain to emergency responders, but otherwise denies weakness, numbness or tingling in his extremities. His vitals on presentation to the ED are HR 90, BP 140/80, RR 20, SpO2 98%. What is the most appropriate next step upon presentation to the emergency room?
Answer is C


2026-02-03 23:56:36.063 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 23:56:36.064 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 23:56:36.064 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 23:56:36.065 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 23:56:36.066 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Running for question 99: A 43-year-old man with a history of schizophrenia, currently controlled with medication, comes in for an appointment with his internist. He is concerned about abnormal discharge from both nipples over the past 3 months. The fluid is white, and he wonders if it could be milk. On further review of systems, he endorses a diminished sexual drive. The physician suspects that one of the patient's medications may be the culprit for these symptoms. Which of the following medications is NOT likely to be the cause?
Answer is C
Accuracy of Run 3 is 0.9000


##### Result of grok-4-1-fast-non-reasoning-latest

In [18]:
import httpx

model = "grok-4-1-fast-non-reasoning-latest"

client = OpenAI(
    api_key=os.environ.get("GROK_KEY"),
    base_url="https://api.x.ai/v1",
    timeout=httpx.Timeout(3600.0),
)


df_structured_tmp = df_structured.copy(deep=True)

for run_number in range(1, number_of_trial + 1):

    for i in range(number_of_question):

        start_time = time.perf_counter()

        question = df_structured_tmp.loc[i, "question"]
        option_A = df_structured_tmp.loc[i, "option_A"]
        option_B = df_structured_tmp.loc[i, "option_B"]
        option_C = df_structured_tmp.loc[i, "option_C"]
        option_D = df_structured_tmp.loc[i, "option_D"]
        option_E = df_structured_tmp.loc[i, "option_E"]
        # option_F = df_structured_tmp.loc[i, "option_F"]


        answer = client.responses.create(
            model="grok-4-1-fast-non-reasoning-latest",
            # model="grok-4-1-fast-reasoning-latest",
            input=[
                {"role": "system", "content": prompt_md},
                {"role": "user", "content": f"""Question: {question}
                                Options:
                                A. {option_A}
                                B. {option_B}
                                C. {option_C}
                                D. {option_D}
                                E. {option_E}
                                """},
            ],
            store=False,
            max_output_tokens=1,
            temperature=0
        ).output_text


        end_time = time.perf_counter()


        if len(answer) != 1 or answer not in VALID_OPTIONS:
            answer = None

        df_structured_tmp.loc[i, f"LLM_output_run_{run_number}"] = answer
        df_structured_tmp.loc[i, f"time_taken_run_{run_number}"] = (
            end_time - start_time
        )


    df_structured_tmp[f"is_correct_run_{run_number}"] = (
    df_structured_tmp["answer_idx"].astype(str).str.strip().str.lower() == df_structured_tmp[f"LLM_output_run_{run_number}"].astype(str).str.strip().str.lower())


    accuracy = df_structured_tmp[f"is_correct_run_{run_number}"].mean()
    print(f"Accuracy of Run {run_number} is {accuracy:.4f}")

df_structured_grok4 = df_structured_tmp.copy(deep=True)
df_structured_grok4["model"] = model



Accuracy of Run 1 is 0.7400
Accuracy of Run 2 is 0.7400
Accuracy of Run 3 is 0.7400


In [ ]:
# df_total = pd.concat([df_structured_llama_70b_versatile,  df_structured_gemini, df_structured_gpt_5_nano, df_structured_gpt_41_nano, df_structured_gpt_4o_mini, df_structured_biochatter_41_nano, df_structured_biochatter_5_nano, df_structured_biochirp])
# df_total.reset_index(inplace=True, drop=True)
# df_total#.head()

##### Merge all model result

In [32]:
df_total = pd.concat([df_structured_biochirp, df_structured_gpt_5_mini, df_structured_gpt_41_nano, df_structured_gpt_5_nano, df_structured_llama_70b_versatile, df_structured_gemini, df_structured_biochatter_41_nano, df_structured_biochatter_5_nano, df_structured_grok4])
df_total.reset_index(inplace=True, drop=True)
df_total#.head()

,question,answer,meta_info,answer_idx,option_A,option_B,option_C,option_D,option_E,LLM_output_run_1,time_taken_run_1,is_correct_run_1,LLM_output_run_2,time_taken_run_2,is_correct_run_2,LLM_output_run_3,time_taken_run_3,is_correct_run_3,model
0,A junior orthopaedic surgery resident is compl...,Tell the attending that he cannot fail to disc...,step1,C,Disclose the error to the patient but leave it...,Disclose the error to the patient and put it i...,Tell the attending that he cannot fail to disc...,Report the physician to the ethics committee,Refuse to dictate the operative report,B,9.846212,False,B,10.747194,False,B,8.529797,False,biochirp
1,A 67-year-old man with transitional cell carci...,Cross-linking of DNA,step1,E,Inhibition of thymidine synthesis,Inhibition of proteasome,Hyperstabilization of microtubules,Generation of free radicals,Cross-linking of DNA,E,13.073798,True,E,14.403638,True,E,12.175809,True,biochirp
2,Two weeks after undergoing an emergency cardia...,Cholesterol embolization,step2&3,C,Renal papillary necrosis,Allergic interstitial nephritis,Cholesterol embolization,Eosinophilic granulomatosis with polyangiitis,Polyarteritis nodosa,C,10.284792,True,C,9.902983,True,C,10.117933,True,biochirp
3,A 39-year-old woman is brought to the emergenc...,"Lactose-fermenting, gram-negative rods forming...",step1,D,"Coagulase-positive, gram-positive cocci formin...","Encapsulated, gram-negative coccobacilli formi...","Spore-forming, gram-positive bacilli forming y...","Lactose-fermenting, gram-negative rods forming...","Gamma-hemolytic, gram-positive cocci forming g...",D,49.399382,True,B,48.024466,False,D,41.477144,True,biochirp
4,A 35-year-old man comes to the physician becau...,Ketotifen eye drops,step2&3,B,Erythromycin ointment,Ketotifen eye drops,Warm compresses,Fluorometholone eye drops,Latanoprost eye drops,B,12.064981,True,B,9.114478,True,B,13.115890,True,biochirp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,A 40-year-old man presents to his primary-care...,MR angiography of the brain,step2&3,D,Colonoscopy,Esophagogastroduodenoscopy,Liver function tests,MR angiography of the brain,Serum amylase and lipase,D,1.415586,True,D,0.570659,True,D,0.555727,True,grok-4-1-fast-non-reasoning-latest
896,A 17-year-old female is brought to the emergen...,Autosomal dominant,step1,A,Autosomal dominant,Autosomal recessive,Autosomal partial dominance,X-linked dominant,X-linked recessive,A,2.494222,True,A,0.532670,True,A,0.598829,True,grok-4-1-fast-non-reasoning-latest
897,A 26-year-old male presents to his primary car...,Antigenic variation,step1,B,Exotoxin release,Antigenic variation,Polysaccharide capsule,Bruton's agammaglobulinemia,Lack of necessary vaccination,B,1.937562,True,B,0.569441,True,B,0.532087,True,grok-4-1-fast-non-reasoning-latest
898,A 37-year-old man with no significant past med...,Cervical immobilization,step2&3,C,Lateral cervical film,Three view cervical spine series,Cervical immobilization,IV methylprednisolone,Observation overnight,C,0.557496,True,C,0.542686,True,C,0.539543,True,grok-4-1-fast-non-reasoning-latest


In [20]:
df_total["model"].value_counts()

model
biochirp                              100
gpt-5-mini                            100
gpt-4.1-nano                          100
gpt-5-nano                            100
llama-3.3-70b-versatile               100
gemini-2.5-flash-lite                 100
biochatter(gpt-4.1-nano)              100
biochatter(gpt-5-nano)                100
grok-4-1-fast-non-reasoning-latest    100
Name: count, dtype: int64

In [33]:
df_total.to_excel("Total_result_mcq.xlsx", index=False)